# 01 — Hypothesis Mapping

This notebook walks each core hypothesis (H1–H7) to the module and score that encodes it, and shows the score responding to inputs. It is the executable companion to `docs/hypothesis_matrix.md` and `docs/terminology_translation.md`.

The point: every slogan becomes a number you can move.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
from orme_lab.config import ModelThresholds
from orme_lab.elements import get_element, core_screen_elements
TH = ModelThresholds()

## H1 — unusual/high-spin configurations → `spin_polarization_score`

High-spin = maximized unpaired electrons. A closed shell (Pd, d10) scores 0; an element with a half-filled-ish d shell scores high.

In [ ]:
from orme_lab.spin_states import high_spin_state, low_spin_state, spin_polarization_score
for e in core_screen_elements():
    hs, ls = high_spin_state(e), low_spin_state(e)
    print(f'{e.symbol:2}  high-spin unpaired={hs.unpaired_electrons}  '
          f'score(hs)={spin_polarization_score(hs):.2f}  score(ls)={spin_polarization_score(ls):.2f}')

## H2/H3/H20 — density anisotropy & the 'rice-bean' shape → `electron_density`

More unpaired spins → more prolate (bean-like) density. The 'rice-bean' band is a *middle* range of anisotropy (see thresholds).

In [ ]:
from orme_lab.electron_density import ricebean_score
for e in core_screen_elements():
    score, bean = ricebean_score(high_spin_state(e), TH)
    print(f'{e.symbol:2}  anisotropy={score:.3f}  rice-bean band? {bean}')
print(f'\nrice-bean band = [{TH.anisotropy_ricebean_low}, {TH.anisotropy_ricebean_high}]')

## H4/H5 — inter-unit coupling & the isolation problem → `coupling`

The decisive gate. A monomer is isolated (coupling 0); connectivity grows with cluster compactness.

In [ ]:
from orme_lab.coupling import inter_unit_coupling_score, is_electronically_isolated
from orme_lab.geometry import make_monomer, make_dimer, make_linear_chain, make_compact_cluster
pt = get_element('Pt')
for geom in [make_monomer(pt), make_dimer(pt), make_linear_chain(pt, 4), make_compact_cluster(pt, 13)]:
    c = inter_unit_coupling_score(geom, TH)
    print(f'{geom.label:10}  coupling={c:.3f}  isolated? {is_electronically_isolated(c, TH)}')

## H7/H19 — magnetic-field response → `magnetic_field`

For a superconducting candidate a field *suppresses* the state (parabolic `1-(H/Hc)^2`); for a magnetic high-spin state a field can *stabilize* it. Two directions, one physics.

In [ ]:
from orme_lab.magnetic_field import magnetic_field_suppression_factor
Hc = 3.0  # toy critical field (tesla)
for H in [0.0, 0.5, 1.0, 2.0, 3.0, 4.0]:
    print(f'H={H:.1f} T  survival factor = {magnetic_field_suppression_factor(H, Hc):.3f}')

## The AND-gate — `superconductivity_plausibility_score`

All five necessary conditions must pass. Killing any one zeroes the score — no averaging, no compensation.

In [ ]:
from orme_lab.superconductivity import superconductivity_plausibility_score
strong = dict(coupling_score=0.8, carrier_proxy=0.8, field_suppression=0.8,
              structural_stability=0.8, observable_signal=0.8, thresholds=TH)
print('all strong :', superconductivity_plausibility_score(**strong).explain())
no_coupling = {**strong, 'coupling_score': 0.0}
print('no coupling:', superconductivity_plausibility_score(**no_coupling).explain())

### Extended hypotheses (H12, H16)

The 'light flows through it' claim is translated (see `docs/terminology_translation.md`) as **electromagnetic/polaritonic coherence**, not electrons-becoming-photons. That is a roadmap module (`electromagnetic_coherence.py`), not yet implemented — noted here so the mapping is complete.